# Driver Drowsiness Detection Using Deep Learning Techniques
**Week 2 — Data Preprocessing & Augmentation Pipeline**  
NTCC Project | Amity School of Engineering & Technology | May 2026

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')

DRIVE_PATH  = '/content/drive/MyDrive/NTCC_Drowsiness_Project'
DATA_DIR    = f'{DRIVE_PATH}/data'
RESULT_DIR  = f'{DRIVE_PATH}/results'
MODEL_DIR   = f'{DRIVE_PATH}/models'

print(f'TensorFlow : {tf.__version__}')
print(f'OpenCV     : {cv2.__version__}')
print(f'Data dir   : {DATA_DIR}')

In [ ]:
classes = [d for d in os.listdir(DATA_DIR)
           if os.path.isdir(os.path.join(DATA_DIR, d))]
print('Classes:', classes)

for cls in classes:
    count = len([f for f in os.listdir(os.path.join(DATA_DIR, cls))
                 if f.lower().endswith(('.jpg','.jpeg','.png'))])
    print(f'  {cls:20s}: {count} images')

In [ ]:
IMG_SIZE   = 64
BATCH_SIZE = 32

# Map class folder names to labels
LABEL_MAP  = {
    'Closed_Eyes' : 0,
    'Open_Eyes'   : 1,
    'Yawn'        : 2,
    'no_yawn'     : 3
}

print('Image size  :', IMG_SIZE, 'x', IMG_SIZE)
print('Batch size  :', BATCH_SIZE)
print('Label map   :', LABEL_MAP)

In [ ]:
def load_images(data_dir, classes, img_size, label_map):
    X, y = [], []
    corrupt = 0
    for cls in classes:
        cls_path = os.path.join(data_dir, cls)
        label    = label_map.get(cls, -1)
        imgs     = [f for f in os.listdir(cls_path)
                    if f.lower().endswith(('.jpg','.jpeg','.png'))]
        for img_name in imgs:
            img = cv2.imread(os.path.join(cls_path, img_name),
                             cv2.IMREAD_GRAYSCALE)
            if img is None:
                corrupt += 1
                continue
            img = cv2.resize(img, (img_size, img_size))
            X.append(img)
            y.append(label)
    X = np.array(X, dtype='float32') / 255.0
    y = np.array(y, dtype='int32')
    print(f'Loaded  : {len(X)} images')
    print(f'Corrupt : {corrupt}')
    print(f'X shape : {X.shape}')
    print(f'y shape : {y.shape}')
    return X, y

X, y = load_images(DATA_DIR, classes, IMG_SIZE, LABEL_MAP)

In [ ]:
# Verify normalisation
print('Pixel value range — min:', X.min(), ' max:', X.max())
print('Mean pixel value        :', round(X.mean(), 4))
print('Std  pixel value        :', round(X.std(), 4))

In [ ]:
# Check class balance after loading
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    cls_name = [k for k,v in LABEL_MAP.items() if v == u][0]
    print(f'Label {u} ({cls_name:15s}): {c} samples ({c/len(y)*100:.1f}%)')

In [ ]:
# Train / Validation / Test split — 70 / 15 / 15
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'Train : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Val   : {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)')
print(f'Test  : {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)')

In [ ]:
# Reshape for CNN input — add channel dimension
X_train = X_train.reshape(-1, IMG_SIZE, IMG_SIZE, 1)
X_val   = X_val.reshape(-1,   IMG_SIZE, IMG_SIZE, 1)
X_test  = X_test.reshape(-1,  IMG_SIZE, IMG_SIZE, 1)

print('X_train shape:', X_train.shape)
print('X_val   shape:', X_val.shape)
print('X_test  shape:', X_test.shape)

In [ ]:
# One-hot encode labels
from tensorflow.keras.utils import to_categorical
NUM_CLASSES = len(LABEL_MAP)

y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat   = to_categorical(y_val,   NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

print('y_train_cat shape:', y_train_cat.shape)
print('y_val_cat   shape:', y_val_cat.shape)
print('y_test_cat  shape:', y_test_cat.shape)

In [ ]:
# Data Augmentation — applied only to training set
train_datagen = ImageDataGenerator(
    rotation_range     = 10,
    width_shift_range  = 0.1,
    height_shift_range = 0.1,
    horizontal_flip    = True,
    brightness_range   = [0.8, 1.2],
    zoom_range         = 0.1
)

val_datagen = ImageDataGenerator()  # no augmentation on val/test

print('Augmentation config:')
print('  rotation_range    : 10 degrees')
print('  width_shift_range : 10%')
print('  height_shift_range: 10%')
print('  horizontal_flip   : True')
print('  brightness_range  : 0.8 - 1.2')
print('  zoom_range        : 10%')

In [ ]:
# Visualise augmentation effect on one sample
sample_img  = X_train[0].reshape(1, IMG_SIZE, IMG_SIZE, 1)
sample_gen  = ImageDataGenerator(
    rotation_range=10, horizontal_flip=True,
    brightness_range=[0.7, 1.3], zoom_range=0.15)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('Original (top) vs Augmented versions (bottom)', fontsize=12)

for i in range(8):
    axes[0][i].imshow(X_train[i].reshape(IMG_SIZE, IMG_SIZE),
                      cmap='gray', vmin=0, vmax=1)
    axes[0][i].axis('off')
    aug = next(sample_gen.flow(sample_img, batch_size=1))[0]
    axes[1][i].imshow(aug.reshape(IMG_SIZE, IMG_SIZE),
                      cmap='gray', vmin=0, vmax=1)
    axes[1][i].axis('off')

plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/augmentation_preview.png', dpi=150)
plt.show()

In [ ]:
# Pixel distribution before and after normalisation comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(X_train.flatten(), bins=50, color='#1E88E5', alpha=0.8, edgecolor='white')
axes[0].set_title('Pixel Distribution (after normalisation)')
axes[0].set_xlabel('Pixel value (0-1)')
axes[0].set_ylabel('Frequency')

axes[1].bar(range(NUM_CLASSES),
            [np.sum(y_train == i) for i in range(NUM_CLASSES)],
            color=['#E53935','#1E88E5','#FF8F00','#43A047'],
            edgecolor='white')
axes[1].set_title('Class distribution in training set')
axes[1].set_xticks(range(NUM_CLASSES))
axes[1].set_xticklabels([k for k in LABEL_MAP.keys()], rotation=15, fontsize=9)
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(f'{RESULT_DIR}/preprocessing_analysis.png', dpi=150)
plt.show()

In [ ]:
# Save processed arrays to Drive
np.save(f'{DRIVE_PATH}/data/X_train.npy', X_train)
np.save(f'{DRIVE_PATH}/data/X_val.npy',   X_val)
np.save(f'{DRIVE_PATH}/data/X_test.npy',  X_test)
np.save(f'{DRIVE_PATH}/data/y_train.npy', y_train_cat)
np.save(f'{DRIVE_PATH}/data/y_val.npy',   y_val_cat)
np.save(f'{DRIVE_PATH}/data/y_test.npy',  y_test_cat)

print('Saved to Drive:')
for fname in ['X_train','X_val','X_test','y_train','y_val','y_test']:
    path = f'{DRIVE_PATH}/data/{fname}.npy'
    size = os.path.getsize(path) / 1e6
    print(f'  {fname}.npy — {size:.1f} MB')

In [ ]:
# Week 2 summary
print('='*50)
print('WEEK 2 SUMMARY')
print('='*50)
print(f'Dataset source  : Kaggle — dheerajperumandla/drowsiness-dataset')
print(f'Total images    : {len(X)}')
print(f'Classes         : {list(LABEL_MAP.keys())}')
print(f'Image input size: {IMG_SIZE}x{IMG_SIZE} grayscale')
print(f'Normalisation   : Divided by 255 (range 0-1)')
print(f'Train samples   : {X_train.shape[0]}')
print(f'Val   samples   : {X_val.shape[0]}')
print(f'Test  samples   : {X_test.shape[0]}')
print(f'Augmentation    : rotation, flip, brightness, zoom')
print(f'Split saved     : .npy files in Drive/data/')
print('='*50)
print('Next (Week 3)   : CNN architecture design + first training run')

In [ ]:
import shutil
from datetime import datetime

REPO_PATH = '/content/Driver-Drowsiness-Detection-Using-Deep-Learning-Techniques'
NOTEBOOK  = 'Week2_Preprocessing.ipynb'

shutil.copy(f'/content/{NOTEBOOK}', f'{REPO_PATH}/notebooks/{NOTEBOOK}')
os.chdir(REPO_PATH)
os.system('git add .')
os.system(f'git commit -m "Week 2: Preprocessing & Augmentation — {datetime.now().strftime("%d %b %Y")}'+'"')
print(os.popen('git push 2>&1').read())